In [4]:
from pydantic import BaseModel


In [8]:
# define the model
class PatientData(BaseModel):
    name: str
    age: int

def add_patient_data(patient:PatientData):
    print(patient.name)
    print(patient.age)

# instantiate the model
patient_data = {"name":"Ritesh", "age":22}

# pass this data to PatientData
patient = PatientData(**patient_data) # since it is dict so we have to unpace it into key and value pair.

add_patient_data(patient)


Ritesh
22


In [14]:
from typing import List, Dict

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]  # all allergies should be in string 
    contact_info: Dict[str,str] # Ex: {"email": "id@gmail.com", "phone":"949858395"}

def add_patient_data(patient:PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.contact_info)

# instantiate the model
patient_data = {"name":"Ritesh", "age":22,"weight": 55, "married":"True","allergies":["peanuts", "shellfish"],"contact_info": {"email": "ritesh@gmail.com","phone": "99485835"}}

# pass this data to PatientData
patient = PatientData(**patient_data) # since it is dict so we have to unpace it into key and value pair.

add_patient_data(patient)


Ritesh
22
55.0
{'email': 'ritesh@gmail.com', 'phone': '99485835'}


### To make any one optional


In [15]:
from typing import List, Dict, Optional

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]  
    contact_info: Optional[Dict[str,str]] = None  # default will be none

def add_patient_data(patient:PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.contact_info)

patient_data = {"name":"Ritesh", "age":22,"weight": 55, "married":"True","allergies":["peanuts", "shellfish"]}

patient = PatientData(**patient_data) # since it is dict so we have to unpace it into key and value pair.

add_patient_data(patient)


Ritesh
22
55.0
None


### Constraints in pydantic

In [16]:
from pydantic import BaseModel, Field

class Patient(BaseModel):
    name: str
    age: int = Field(gt=0, lt=120)
    weight: float = Field(gt=0)

In [ ]:
# It will work
patient = Patient(
    name="Ritesh",
    age=22,
    weight=55
)

In [ ]:
# It will not work
patient = Patient(
    name="Ritesh",
    age=-5,
    weight=55
)

ValidationError: 1 validation error for Patient
age
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [21]:
# String validation
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(min_length=3, max_length=20)
    email: str

user = User(
    username="Rit",
    email="ritesh@gmail.com"
)

In [22]:
# String validation
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(min_length=3, max_length=20)
    email: str

user = User(
    username="Ri",
    email="ritesh@gmail.com"
)

ValidationError: 1 validation error for User
username
  String should have at least 3 characters [type=string_too_short, input_value='Ri', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short

### Field Validator

In [27]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    name: str
    email:str
    age: int

    @field_validator("email")  # decorator 1
    @classmethod             # decorator 2 (as this is the method of above class 
                             # so used classMethod)
    def validate_age(cls, value):  # it will take obj of class(cls) and value means email address
        valid_domain=['gmail.com', "custom.com"]
        domain_name = value.split('@')[-1]
        if domain_name not in valid_domain:
            raise ValueError("It is not a valid domain")

        return value


user = User(name="Ritesh",email="ritesh@custom.com", age=22)

print(user)

name='Ritesh' email='ritesh@custom.com' age=22


In [29]:
from pydantic import BaseModel, field_validator


class User(BaseModel):
    name: str
    email: str
    age: int

    @field_validator("email")
    @classmethod
    def validate_email(cls, value):

        valid_domain = ["gmail.com", "custom.com"]

        domain_name = value.split("@")[-1]

        if domain_name not in valid_domain:
            raise ValueError("It is not a valid domain")

        return value


user = User(
    name="Ritesh",
    email="ritesh@customn.com",
    age=22
)

print(user)

ValidationError: 1 validation error for User
email
  Value error, It is not a valid domain [type=value_error, input_value='ritesh@customn.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

### Computed Fields in Pydantic
- A computed field is a field whose value is calculated from other fields rather than being directly provided by the user.
- For example, if you have price = 100 and quantity = 3

- You can automatically calculate: total = 300